In [ ]:
using Pkg
Pkg.activate(".")
Pkg.develop(path="..")

isCuda = try
    success(`nvidia-smi`)
catch
    false
end
if isCuda
    println("CUDA is available, using GPU acceleration.")
    using CUDA
end

using bslLD
bslLD.greet()

if isCuda
    println("Setting backend to CUDA.")
    bslLD.use_cuda!()
else
    println("CUDA not available, using CPU.")
end


In [ ]:
# k = (kx, 0, 0),  B0 = z-hat  (grid.Bdir = 3)
# Only Bz and (Ex, Ey) evolve; Ez = 0 and Pi_diff_z = 0 exactly for this geometry.

beta_i  = 0.1     # ion beta
mu      = 100.0    # mass ratio m_i / m_e
epsilon = 1e-6     # perturbation amplitude

Lx   = 20pi
Nx   = 64
Nv   = 32
vmax = 4.0

grid    = bslLD.Grid([0.0, -vmax, -vmax], [Lx, vmax, vmax], [Nx, Nv, Nv], 1, 1.0, 3)
simTime = bslLD.SimulationTime(0.025, 30.0; gyro_frequency=1.0)


In [ ]:
# Ion distribution: Maxwellian in (vx, vy) with a random density perturbation in x
initFuncx(x) = 1.0 + epsilon * randn()
initFuncv(v) = exp(-v^2 / 2) / sqrt(2pi)

f_i = bslLD.Distribution(grid, 0.0;
    initFuncx = initFuncx,
    initFuncv = initFuncv,
)


In [ ]:
# Initialize FieldSolution: small sinusoidal Ex perturbation, zero dB
x = collect(grid.xaxes[1])
E0 = bslLD.VectorField([
    bslLD.ScalarField(epsilon .* sin.(2pi / Lx .* x)),
    bslLD.ScalarField(zeros(Nx)),
    bslLD.ScalarField(zeros(Nx)),
])
sol = bslLD.FieldSolution(E0, bslLD.zero_vectorfield3(grid), bslLD.background_field(grid))


In [ ]:
# Compute perpendicular ion current J_i_perp in the LAB frame from a 1D2V distribution.
# The distribution is stored in rotating-frame velocity coordinates: v_lab = R(Bdir, phi) * v_grid
# (advectorCart.jl XShiftContext: xdisp = sum_dv R(Bdir,phi)[dir,dv] * v_grid[dv]).
# Therefore J_lab = R(Bdir, phi) * J_grid  where  J_grid = ∫ v_grid f dv_grid.
# For Bdir=3:  R(3,phi) = [cos φ  -sin φ; sin φ  cos φ]  (column-major SMatrix).
function compute_J_perp(f_i, grid, phi)
    Nx_  = length(grid.xaxes[1])
    vx   = collect(grid.vaxes[1])
    vy   = collect(grid.vaxes[2])
    dvx  = grid.delta[2]
    dvy  = grid.delta[3]

    data = Array(f_i.data)   # shape (Nx, Nvx, Nvy)

    # moments in rotating frame
    Jx_rot = reshape(sum(data .* reshape(vx, 1, :, 1), dims=(2, 3)), Nx_) .* (dvx * dvy)
    Jy_rot = reshape(sum(data .* reshape(vy, 1, 1, :), dims=(2, 3)), Nx_) .* (dvx * dvy)

    # rotate to lab frame: J_lab = R(3, phi) * J_rot
    c, s   = cos(phi), sin(phi)
    Jx_lab = c .* Jx_rot .- s .* Jy_rot
    Jy_lab = s .* Jx_rot .+ c .* Jy_rot

    return bslLD.VectorField([
        bslLD.ScalarField(bslLD.backend_array(Jx_lab)),
        bslLD.ScalarField(bslLD.backend_array(Jy_lab)),
        bslLD.ScalarField(bslLD.backend_array(zeros(Nx_))),
    ])
end


In [ ]:
# Strang-splitting step: V/2 - X - field-update - V/2
# Pi_diff_z = 0 exactly for k=(kx,0,0) with no z-variation
const em_solver = bslLD.SemiImplicitEMSolver(beta_i, mu)
const Pi_zero   = bslLD.zero_vectorfield3(grid)

function stepStrang_EM!(f_i, sol, grid, simTime)
    phase_start = simTime.phase
    Ω = simTime.gyro_frequency

    # V half-step at phase(t)
    simTime.fraction_dt = 0.5
    bslLD.advectV!(f_i, grid, simTime, sol.E)

    # X full-step at phase(t + dt/2)
    simTime.phase = phase_start + Ω * simTime.dt * 0.5
    simTime.fraction_dt = 1.0
    bslLD.advectX!(f_i, grid, simTime)

    # Compute moments at the mid-step phase and update fields
    # simTime.phase is now phase_start + Ω*dt/2, which is the correct rotation angle
    J_perp  = compute_J_perp(f_i, grid, simTime.phase)
    moments = bslLD.Moments(bslLD.compute_density(f_i, grid), J_perp, Pi_zero)
    bslLD.solve_fields!(sol, moments, grid, em_solver, simTime.dt)

    # V half-step at phase(t + dt) with updated E
    simTime.phase = phase_start + Ω * simTime.dt
    simTime.fraction_dt = 0.5
    bslLD.advectV!(f_i, grid, simTime, sol.E)
    simTime.fraction_dt = 1.0

    simTime.phase = phase_start   # restore so advance!() applies the correct increment
end


In [ ]:
mutable struct Diag
    Ex :: Vector
    Bz :: Vector
    t  :: Vector{Float64}
end
Diag() = Diag([], [], Float64[])

function record!(d, sol, simTime)
    simTime.step % 4 == 0 || return
    push!(d.Ex, copy(Array(sol.E[1].data)))
    push!(d.Bz, copy(Array(sol.B[3].data)))
    push!(d.t,  simTime.current_T)
end


In [ ]:
bslLD.ProgressMeter.ijulia_behavior(:clear)
diag = Diag()

while bslLD.continue_advection(simTime, true)
    stepStrang_EM!(f_i, sol, grid, simTime)
    record!(diag, sol, simTime)
    bslLD.advance!(simTime)
end


In [ ]:
using FFTW, DSP, CairoMakie

data     = transpose(hcat(diag.Ex...))   # (Nt, Nx)
dt_diag  = 4 * simTime.dt
omega    = fftfreq(size(data, 1), 1 / dt_diag) .* 2pi
k        = fftfreq(Nx, 1 / grid.delta[1]) .* 2pi

nw = size(data, 1) ÷ 4
nk = Nx ÷ 2

w_win    = kaiser(size(data, 1), 3)
k_win    = kaiser(Nx, 3)
windowed = data .* w_win .* k_win'

spec = log.(abs.(fft(windowed))[1:nw, 1:nk] .+ 1e-30)

fig, ax, hm = heatmap(k[1:nk], omega[1:nw], spec',
    axis = (xlabel = "k", ylabel = "ω", title = "Eₓ spectrum — EM, k=(kx,0,0)"))
Colorbar(fig[1, 2], hm)
fig
